# From raw/nis/20* we will extract data from our .DAT file to generate CSV's and merge into one

### Let's do a quick demo before building the function: ```read_nis```
Using
- raw/nis/2015/NIS-PUF15.SAS
- raw/nis/2015/NISPUF15.DAT

In [1]:
with open('../raw/nis/2015/NIS-PUF15.SAS', "r", encoding="latin1") as f:
        sas_text = f.read()

### We need to parse value and their numerical reps
Example
```
value SEX
. = "MISSING"
1 = "MALE"
2 = "FEMALE"
77 = "DON'T KNOW"
99 = "REFUSED"
```

### Read the .DAT file

In [2]:
import pandas as pd

df = pd.read_fwf(
    "../raw/nis/2015/NISPUF15.DAT",
    colspecs=[(176, 177)],
    names=["SEX"]
)

sex_map = {
    1: "MALE",
    2: "FEMALE",
    77: "DON'T KNOW",
    99: "REFUSED",
}

df["SEX_label"] = df["SEX"].map(sex_map)

df.head()

,SEX,SEX_label
0,1,MALE
1,1,MALE
2,1,MALE
3,2,FEMALE
4,2,FEMALE


### According to ChatGPT, these are the most important columns for vaccine coverage:
| Column         | Level          | What it means                              | Why you need it                                                                     |
| -------------- | -------------- | ------------------------------------------ | ----------------------------------------------------------------------------------- |
| **`SEQNUMC`**  | Child          | Unique child identifier                    | Identifies the individual child/record                                              |
| **`SEQNUMHH`** | Household      | Household identifier                       | Identifies which household the child belongs to; used in survey design              |
| **`STRATUM`**  | Sampling group | Survey sampling stratum                    | Identifies the sampling group; needed for correct SEs/CIs                           |
| **`PROVWT`**   | Child          | Provider-phase survey weight               | Determines how much the child contributes to population estimates                   |
| **`P_UTDMCV`** | Child          | Measles vaccination status                 | `1` = received ≥1 qualifying measles-containing vaccination; `0` = did not          |
| **`P_NUMMMR`** | Child          | Number of measles-containing vaccine doses | Shows how many provider-reported measles-containing vaccinations the child received |
| **`STATE`**    | Geography      | State code                                 | Lets you calculate vaccination coverage by state                                    |
| **`YEAR`**     | Time           | Survey year                                | Lets you calculate and compare coverage over time                                   |

In [3]:
import pandas as pd

colspecs = [
    (6, 11),      # SEQNUMHH
    (12, 32),     # PROVWT_D
    (90, 94),     # STRATUM
    (94, 98),     # YEAR
    (182, 184),   # STATE
    (253, 254),   # P_UTDMCV
]

names = [
    "SEQNUMHH",
    "PROVWT_D",
    "STRATUM",
    "YEAR",
    "STATE",
    "P_UTDMCV",
]

df = pd.read_fwf(
    "../raw/nis/2015/NISPUF15.DAT",
    colspecs=colspecs,
    names=names,
    na_values=["."]
)

df.head()

,SEQNUMHH,PROVWT_D,STRATUM,YEAR,STATE,P_UTDMCV
0,1,77.861750,2017,2015,42,1.0
1,2,NaN,2072,2015,15,NaN
2,3,73.609547,2019,2015,54,1.0
3,4,NaN,2002,2015,25,NaN
4,5,141.333362,2075,2015,16,1.0


In [4]:
len(df)

27592

In [5]:
df.dtypes

SEQNUMHH      int64
PROVWT_D    float64
STRATUM       int64
YEAR          int64
STATE         int64
P_UTDMCV    float64
dtype: object

In [6]:
assert False

AssertionError: 

## Let's start working on this function comprised of four steps:
1. get column row from .SAS
2. parse data and turn to df
3. translate STATE
4. concat df's
4. save total_df as csv

In [32]:
import us
import pandas as pd

def translate_state(sas_text):

    # Get STATE value block
    match = re.search(
        r"value\s+STATE\b(.*?);",
        sas_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    state_block = match.group(1)

    # Extract number = "STATE NAME"
    pairs = re.findall(
        r'(\d+)\s*=\s*"([^"]+)"',
        state_block
    )

    # these are exceptions since the US library doesn't pick up on them
    name_fixes = {
        "DISTRICT OF COLUMBIA": "DC",
        "U.S.  VIRGIN ISLANDS": "VI",
    }

    state_map = {}

    for code, state in pairs:

        if state in name_fixes:
            state_map[int(code)] = name_fixes[state]
            continue

        result = us.states.lookup(state)

        if result:
            state_map[int(code)] = result.abbr
        else:
            print(f"Not found: {code} = {state}")

    return state_map

In [33]:
import re

def parse_dat_to_csv(names, sas_file_path, dat_file_path):

    # names is array of the core columns to extract

    # parse col locations and save in array
    colspecs = []
    with open(sas_file_path, "r", encoding="latin1") as f:
        sas_text = f.read()


        for name in names:
            match = re.search(
                rf"@\d+\s+({name})\s+\$?\d+(?:\.\d*)?",
                sas_text
            )

            col_loc = match.group()

            numbers = re.findall(r"\d+", col_loc)

            start = int(numbers[0]) - 1
            end = start + int(numbers[1])

            colspecs.append((start, end))

    df = pd.read_fwf(
        dat_file_path,
        colspecs=colspecs,
        names=names,
        na_values=["."]
    )

    # translate STATE
    state_map = translate_state(sas_text)
    df["STATE"] = df["STATE"].map(state_map)

    # return df
    return df


In [34]:
from pathlib import Path

names = [
    "SEQNUMC",
    "SEQNUMHH",
    r"PROVWT_\w+",
    "STRATUM",
    "YEAR",
    "STATE",
    "P_UTDMCV",
    "P_NUMMMR"
]

total_df = pd.DataFrame(columns=names)

folder = Path("../raw/nis")
for subfolder in sorted(folder.iterdir()):
    if subfolder.is_dir():

        dat_file = None
        sas_file = None
        print(f'Working on: {subfolder}')
        for file in subfolder.iterdir():
            if file.is_file():
                if file.suffix.upper() == ".DAT":
                    dat_file = str(file)

                if file.suffix.upper() == ".SAS":
                    sas_file = str(file)

                if dat_file and sas_file:
                    df = parse_dat_to_csv(names, sas_file, dat_file)

                    total_df = pd.concat([total_df, df], ignore_index=True)

Working on: ../raw/nis/2015
Working on: ../raw/nis/2016
Working on: ../raw/nis/2017
Working on: ../raw/nis/2018
Working on: ../raw/nis/2019
Working on: ../raw/nis/2020
Working on: ../raw/nis/2021
Working on: ../raw/nis/2022
Working on: ../raw/nis/2023
Working on: ../raw/nis/2024


In [35]:
total_df.rename(
    columns=lambda x: "PROVWT" if x.startswith("PROVWT_") else x,
    inplace=True
)
total_df.head()

,SEQNUMC,SEQNUMHH,PROVWT,STRATUM,YEAR,STATE,P_UTDMCV,P_NUMMMR
0,11,1,77.86175,2017,2015,PA,1.0,1.0
1,21,2,NaN,2072,2015,HI,NaN,NaN
2,31,3,73.609547,2019,2015,WV,1.0,1.0
3,41,4,NaN,2002,2015,MA,NaN,NaN
4,51,5,141.333362,2075,2015,ID,1.0,1.0


In [36]:
len(total_df)

326068

# Let's further refind total_df to ensure it works well with our measles incidence data ```rada_notebooks/nis_measles_vacc_coverage.csv (Primary coverage)```

In [37]:
total_df.rename(columns={'STATE': 'state', 'YEAR': 'year'}, inplace=True)

In [38]:
total_df.head()

,SEQNUMC,SEQNUMHH,PROVWT,STRATUM,year,state,P_UTDMCV,P_NUMMMR
0,11,1,77.86175,2017,2015,PA,1.0,1.0
1,21,2,NaN,2072,2015,HI,NaN,NaN
2,31,3,73.609547,2019,2015,WV,1.0,1.0
3,41,4,NaN,2002,2015,MA,NaN,NaN
4,51,5,141.333362,2075,2015,ID,1.0,1.0


### Below we group to calculate weigted coverage for each year and state
- P_UTDMCV: child is vaccinated?
- PROVWT: how much the child has contributed to the coverage

In [39]:
from statsmodels.stats.weightstats import DescrStatsW
import numpy as np

def weighted_coverage(group):
    valid = (
        group["P_UTDMCV"].notna()
        & group["PROVWT"].notna()
        & (group["PROVWT"] > 0)
    )

    if valid.sum() == 0:
        return np.nan

    stats = DescrStatsW(
        group.loc[valid, "P_UTDMCV"],
        weights=group.loc[valid, "PROVWT"],
        ddof=0
    )

    return stats.mean

In [40]:
coverage_df = (
    total_df
    .groupby(["state", "year"])
    .apply(weighted_coverage, include_groups=False)
    .reset_index(name="coverage")
)

In [41]:
coverage_df.head()

,state,year,coverage
0,AK,2015,0.897005
1,AK,2016,0.858291
2,AK,2017,0.895626
3,AK,2018,0.851136
4,AK,2019,0.857868


### Let's also find out how many children were vaccinated

In [42]:
coverage_df = (
    total_df
    .dropna(subset=["P_UTDMCV", "PROVWT"])
    .groupby(["state", "year"])
    .apply(
        lambda x: pd.Series({
            "coverage": DescrStatsW(
                x["P_UTDMCV"],
                weights=x["PROVWT"],
                ddof=0
            ).mean,
            "n": len(x)
        }),
        include_groups=False
    )
    .reset_index()
)

coverage_df["coverage_pct"] = round(coverage_df["coverage"] * 100, 2)
coverage_df.drop(columns='coverage', inplace=True)

In [43]:
coverage_df.head()

,state,year,n,coverage_pct
0,AK,2015,295.0,89.70
1,AK,2016,288.0,85.83
2,AK,2017,251.0,89.56
3,AK,2018,228.0,85.11
4,AK,2019,240.0,85.79


In [45]:
coverage_df.to_csv('../app/data/nis_measles_vacc_coverage.csv', index=False)